In [ ]:
# CELL 1: Project Title and Description
"""
Gemini-powered LangChain Agent for Student Information Management

Day 5 Workshop Assignment - Agentic AI

This notebook demonstrates how an agent dynamically selects and uses tools
based on user questions, rather than following a fixed sequence.
"""


In [ ]:
# CELL 2: Install Required Packages
!pip install langchain langchain-community langchain-google-genai -q


In [ ]:
# CELL 3: Imports
import os
import sqlite3
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent


In [ ]:
# CELL 4: Secure Gemini API Key Configuration
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = api_key

print("API key configured from Colab Secrets")


In [ ]:
# CELL 5: Create SQLite Database with Student Records
import sqlite3

conn = sqlite3.connect('students.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        student_id TEXT PRIMARY KEY,
        name TEXT,
        department TEXT,
        python INTEGER,
        database INTEGER,
        ai INTEGER,
        web INTEGER
    )
''')

students_data = [
    ('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78),
    ('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72),
    ('22CS047', 'Priya', 'Information Technology', 92, 88, 95, 90),
    ('22CS048', 'Arun', 'Information Technology', 55, 60, 58, 62),
    ('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88),
]

cursor.executemany('''
    INSERT OR REPLACE INTO students 
    (student_id, name, department, python, database, ai, web)
    VALUES (?, ?, ?, ?, ?, ?, ?)
''', students_data)

conn.commit()
conn.close()

print("Database created: students.db")


In [ ]:
# CELL 6: Verify Database Contents
import pandas as pd

conn = sqlite3.connect('students.db')
df = pd.read_sql_query("SELECT * FROM students", conn)
conn.close()

print("\nStudent Records:")
print(df.to_string(index=False))


In [ ]:
# CELL 7: Tool - get_student_info
@tool
def get_student_info(student_id: str) -> str:
    """
    Get basic information about a student.
    
    Args:
        student_id: The student ID (e.g., '22CS045')
    
    Returns:
        Student name and department.
    """
    conn = sqlite3.connect('students.db')
    cursor = conn.cursor()
    cursor.execute('SELECT name, department FROM students WHERE student_id = ?', (student_id,))
    result = cursor.fetchone()
    conn.close()
    
    if result:
        name, department = result
        return f"Name: {name}\nDepartment: {department}"
    else:
        return f"Student {student_id} not found."


In [ ]:
# CELL 8: Tool - get_student_marks
@tool
def get_student_marks(student_id: str) -> str:
    """
    Get the marks of a student in all subjects.
    
    Args:
        student_id: The student ID (e.g., '22CS045')
    
    Returns:
        Marks in Python, Database, AI, and Web.
    """
    conn = sqlite3.connect('students.db')
    cursor = conn.cursor()
    cursor.execute('SELECT python, database, ai, web FROM students WHERE student_id = ?', (student_id,))
    result = cursor.fetchone()
    conn.close()
    
    if result:
        python, database, ai, web = result
        return f"Python: {python}\nDatabase: {database}\nAI: {ai}\nWeb: {web}"
    else:
        return f"Student {student_id} not found."


In [ ]:
# CELL 9: Tool - calculator
@tool
def calculator(expression: str) -> str:
    """
    Perform mathematical calculations.
    
    Args:
        expression: A mathematical expression (e.g., '85 + 72 + 90 + 78')
    
    Returns:
        The result of the calculation.
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"


In [ ]:
# CELL 10: Tool - get_passing_rules
@tool
def get_passing_rules() -> str:
    """
    Get the university's passing requirements.
    
    Returns:
        Minimum marks and average requirements for passing.
    """
    return """University Passing Requirements:
- Minimum overall average: 40%
- Minimum mark in each subject: 35%"""


In [ ]:
# CELL 11: Tool - get_class_statistics
@tool
def get_class_statistics(query: str) -> str:
    """
    Get class-level statistics about students.
    
    Can provide:
    - Average marks in each subject (Python, Database, AI, Web)
    - Number of students in each department (Computer Science, IT)
    - Highest and lowest marks in each subject
    - Total number of students
    
    Args:
        query: A question about class statistics
    
    Returns:
        The requested statistic.
    """
    conn = sqlite3.connect('students.db')
    cursor = conn.cursor()
    query_lower = query.lower()
    
    # Average marks
    if "average" in query_lower:
        if "python" in query_lower:
            cursor.execute('SELECT AVG(python) FROM students')
            avg = cursor.fetchone()[0]
            conn.close()
            return f"Average Python mark: {avg:.2f}"
        elif "database" in query_lower or "db" in query_lower:
            cursor.execute('SELECT AVG(database) FROM students')
            avg = cursor.fetchone()[0]
            conn.close()
            return f"Average Database mark: {avg:.2f}"
        elif "ai" in query_lower:
            cursor.execute('SELECT AVG(ai) FROM students')
            avg = cursor.fetchone()[0]
            conn.close()
            return f"Average AI mark: {avg:.2f}"
        elif "web" in query_lower:
            cursor.execute('SELECT AVG(web) FROM students')
            avg = cursor.fetchone()[0]
            conn.close()
            return f"Average Web mark: {avg:.2f}"
    
    # Count by department
    if ("computer science" in query_lower or "cs" in query_lower) and ("count" in query_lower or "how many" in query_lower):
        cursor.execute('SELECT COUNT(*) FROM students WHERE department = "Computer Science"')
        count = cursor.fetchone()[0]
        conn.close()
        return f"Number of Computer Science students: {count}"
    
    if ("information technology" in query_lower or "it" in query_lower) and ("count" in query_lower or "how many" in query_lower):
        cursor.execute('SELECT COUNT(*) FROM students WHERE department = "Information Technology"')
        count = cursor.fetchone()[0]
        conn.close()
        return f"Number of Information Technology students: {count}"
    
    # Highest marks
    if "highest" in query_lower or "maximum" in query_lower or "max" in query_lower:
        if "python" in query_lower:
            cursor.execute('SELECT MAX(python) FROM students')
            max_mark = cursor.fetchone()[0]
            conn.close()
            return f"Highest Python mark: {max_mark}"
        elif "database" in query_lower or "db" in query_lower:
            cursor.execute('SELECT MAX(database) FROM students')
            max_mark = cursor.fetchone()[0]
            conn.close()
            return f"Highest Database mark: {max_mark}"
        elif "ai" in query_lower:
            cursor.execute('SELECT MAX(ai) FROM students')
            max_mark = cursor.fetchone()[0]
            conn.close()
            return f"Highest AI mark: {max_mark}"
        elif "web" in query_lower:
            cursor.execute('SELECT MAX(web) FROM students')
            max_mark = cursor.fetchone()[0]
            conn.close()
            return f"Highest Web mark: {max_mark}"
    
    conn.close()
    return "Query not understood. Try asking about average marks, student counts, or highest marks."


In [ ]:
# CELL 12: Create Gemini Model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

print("Gemini model created")


In [ ]:
# CELL 13: Create LangChain Agent
system_prompt = """You are a student information assistant.

You have access to tools for:
- Student information (name, department)
- Student marks in all subjects
- Mathematical calculations
- University passing requirements
- Class-level statistics

Use only the tools necessary to answer each question.
Use the calculator for numerical calculations.
Use passing rules when determining eligibility.
Do not invent student information or university rules.
Provide clear answers based on tool results."""

tools = [
    get_student_info,
    get_student_marks,
    calculator,
    get_passing_rules,
    get_class_statistics
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Agent created with 5 tools")


In [ ]:
# CELL 14: Test Question 1 - Student Information
question_1 = "What is the name and department of student 22CS045?"

print(f"Q: {question_1}\n")
result = agent.invoke({"messages": [{"role": "user", "content": question_1}]})
print(f"A: {result['messages'][-1].content}\n")


In [ ]:
# CELL 15: Test Question 2 - Student Marks
question_2 = "What are the marks of student 22CS047?"

print(f"Q: {question_2}\n")
result = agent.invoke({"messages": [{"role": "user", "content": question_2}]})
print(f"A: {result['messages'][-1].content}\n")


In [ ]:
# CELL 16: Test Question 3 - Total and Average Marks
question_3 = "What is the total and average mark of student 22CS045?"

print(f"Q: {question_3}\n")
result = agent.invoke({"messages": [{"role": "user", "content": question_3}]})
print(f"A: {result['messages'][-1].content}\n")


In [ ]:
# CELL 17: Test Question 4 - Eligibility Check
question_4 = "Is student 22CS045 eligible to pass according to the university rules?"

print(f"Q: {question_4}\n")
result = agent.invoke({"messages": [{"role": "user", "content": question_4}]})
print(f"A: {result['messages'][-1].content}\n")


In [ ]:
# CELL 18: Challenge Question - Full Student Profile
challenge_q = "I am student 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements."

print(f"Q: {challenge_q}\n")
result = agent.invoke({"messages": [{"role": "user", "content": challenge_q}]})
print(f"A: {result['messages'][-1].content}\n")


In [ ]:
# CELL 19: Extension - Class Statistics
print("Testing class-statistics extension:\n")

stat_q1 = "What is the average AI mark of the class?"
print(f"Q: {stat_q1}")
result = agent.invoke({"messages": [{"role": "user", "content": stat_q1}]})
print(f"A: {result['messages'][-1].content}\n")

stat_q2 = "How many students are in Computer Science?"
print(f"Q: {stat_q2}")
result = agent.invoke({"messages": [{"role": "user", "content": stat_q2}]})
print(f"A: {result['messages'][-1].content}\n")


In [ ]:
# CELL 20: Summary
print("""
KEY OBSERVATIONS:

1. The agent receives a question and a list of available tools.

2. The agent reads tool descriptions (from docstrings) and type hints
   to understand what each tool does.

3. For simple questions, the agent uses one tool.
   For complex questions, it chains multiple tools together.

4. The agent decides which tools to use - this is dynamic, not hardcoded.

5. For eligibility questions, the agent must use multiple tools
   in the right order without explicit instructions to do so.

6. The class-statistics extension shows that new tools can be added
   without changing the agent architecture.
""")


# Student Information Agent

A Gemini-powered LangChain agent that answers questions about student information by dynamically selecting and using appropriate tools.

## Overview

This project demonstrates agentic AI, where an LLM (Gemini) independently decides which tools to use based on a question, rather than following a predefined sequence.

## Technologies

- **Gemini 1.5 Flash** - LLM for decision-making
- **LangChain** - Agent framework
- **SQLite** - Student database
- **Google Colab** - Development environment

## Available Tools

1. **get_student_info** - Returns student name and department
2. **get_student_marks** - Returns marks in 4 subjects (Python, Database, AI, Web)
3. **calculator** - Performs mathematical calculations
4. **get_passing_rules** - Returns university minimum requirements (40% average, 35% per subject)
5. **get_class_statistics** - Returns class-level statistics (averages, counts, highs/lows)

## How the Agent Works


```

User Question

↓

Gemini reads the question and available tools

↓

Decides which tools to call

↓

Calls tools, receives results

↓

Determines if more tools are needed

↓

Returns final answer

```

The agent makes these decisions based on the question and tool descriptions — not a hardcoded sequence.

## Running in Google Colab

1. Open [Google Colab](https://colab.research.google.com)
2. Upload `Student_Agent.ipynb`
3. Add your API key to Colab Secrets:
   - Click the 🔑 icon (Secrets) on the left
   - Click "Create new secret"
   - Name: `GOOGLE_API_KEY`
   - Value: Your Gemini API key from [ai.google.dev](https://ai.google.dev)
4. Run cells in order

## API Key Safety

The API key is stored securely in Google Colab's Secrets manager and never appears in notebook code or output.

## Example Questions

- "What is the name of student 22CS045?" → Uses: `get_student_info`
- "What are the marks of student 22CS047?" → Uses: `get_student_marks`
- "What is the total and average mark of 22CS045?" → Uses: `get_student_marks`, `calculator`
- "Is 22CS045 eligible to pass?" → Uses: `get_student_marks`, `get_passing_rules`, `calculator`
- "What is the average AI mark of the class?" → Uses: `get_class_statistics`

## Database

The notebook creates `students.db` with 5 sample students across Computer Science and Information Technology departments.

## Files

- `Student_Agent.ipynb` - Complete notebook for Google Colab
- `students.db` - SQLite database (created by notebook)
- `README.md` - This file

## Extension: Class Statistics

The `get_class_statistics` tool allows the agent to answer class-level questions:
- "What is the average [subject] mark of the class?"
- "How many students are in [department]?"
- "What is the highest [subject] mark?"

This demonstrates that the same agent architecture scales to aggregate queries without modification.
